# 🎬 Gujarati Faceless YouTube — run it in your browser

No install, no PowerShell. Just run each cell top to bottom with the **▶ play button**.

1. **Setup** — installs everything (ffmpeg, Python packages, the Gujarati font).
2. **Keys** — paste your Anthropic + ElevenLabs keys (typed privately, not saved in the notebook).
3. **Make a video** — type a topic, get an MP4 in your *Aj Katihyawadi* voice.

> Tip: Runtime menu → *Run all* does everything after you've entered keys.

## 1. Setup (run once per session)

In [ ]:
#@title Install ffmpeg, the project, and the Gujarati font
BRANCH = "claude/gujarati-youtube-automation-q33bzm"  #@param {type:"string"}
import os, subprocess, sys

if not os.path.isdir("Claude"):
    !git clone --branch $BRANCH --depth 1 https://github.com/ajayb1-AJ/Claude.git
%cd Claude

print("Installing ffmpeg...")
!apt-get -qq update && apt-get -qq install -y ffmpeg >/dev/null
print("Installing Python packages...")
!pip install -q -r requirements.txt

# Download the Gujarati subtitle font (open internet here, unlike the sandbox).
os.makedirs("assets/fonts", exist_ok=True)
font = "assets/fonts/NotoSansGujarati-Bold.ttf"
if not os.path.exists(font):
    !wget -q -O "$font" "https://github.com/google/fonts/raw/main/ofl/notosansgujarati/static/NotoSansGujarati-Bold.ttf"
print("ffmpeg:", subprocess.run(["which","ffmpeg"],capture_output=True,text=True).stdout.strip())
print("font bytes:", os.path.getsize(font) if os.path.exists(font) else "MISSING")
print("\n✅ Setup done.")

## 2. Enter your API keys
Typed privately (hidden), written to a local `.env`, never printed or committed.

- **Anthropic** (script) — get at console.anthropic.com (needs credits).
- **ElevenLabs** (voice) — use a **Full Access** key.
- Pexels/Pixabay are pre-filled for stock visuals.

In [ ]:
#@title Paste keys, then run
from getpass import getpass

anthropic = getpass("ANTHROPIC_API_KEY (sk-ant-...): ").strip()
eleven = getpass("ELEVENLABS_API_KEY (sk_...): ").strip()

env = f"""ANTHROPIC_API_KEY={anthropic}
ELEVENLABS_API_KEY={eleven}
PEXELS_API_KEY=cPPrmyVBecxheeBLu12wi5ur4AxchcxQMTDPzdMZC0bwVgbrn8uPZEuL
PIXABAY_API_KEY=57300136-17cf9499b0a5136d677341ad0
"""
open(".env", "w").write(env)
print("Saved .env. Verifying keys live...\n")
!python check_setup.py --live

## 3. Make a video (no upload — for review)

In [ ]:
#@title Generate one video
TOPIC = "\u0ab8\u0abe\u0a9a\u0ac0 \u0aae\u0ab9\u0ac7\u0aa8\u0aa4\u0aa8\u0ac1\u0a82 \u0aab\u0ab3"  #@param {type:"string"}
!python main.py --topic "$TOPIC" --no-upload

In [ ]:
#@title Watch the result here
import glob, os
from base64 import b64encode
from IPython.display import HTML, display
mp4 = sorted(glob.glob("output/*/final.mp4"), key=os.path.getmtime)[-1]
print("Newest video:", mp4, "(", round(os.path.getsize(mp4)/1e6,2), "MB )")
data = b64encode(open(mp4,"rb").read()).decode()
display(HTML(f'<video width=420 controls><source src="data:video/mp4;base64,{data}" type="video/mp4"></video>'))

In [ ]:
#@title (Optional) Download the video to your computer
from google.colab import files
files.download(mp4)